In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/raw_data"))

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. READ (Changed to CSV, added header option)
constructors_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/constructors.csv")

# 2. TRANSFORM
constructors_silver_df = constructors_df \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("constructorRef", "constructor_ref") \
    .withColumn("ingestion_date", current_timestamp()) \
    .drop("url")

# 3. WRITE
constructors_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_constructors")

display(constructors_silver_df)

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. READ (It is a CSV!)
pit_stops_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/pit_stops.csv")

# 2. TRANSFORM
# Rename columns to snake_case
pit_stops_silver_df = pit_stops_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumn("ingestion_date", current_timestamp())

# 3. WRITE
pit_stops_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_pit_stops")

display(pit_stops_silver_df)

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. READ
lap_times_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/lap_times.csv")

# 2. TRANSFORM
lap_times_silver_df = lap_times_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumn("ingestion_date", current_timestamp())

# 3. WRITE
lap_times_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_lap_times")

display(lap_times_silver_df)

In [0]:
# 1. READ
qualifying_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/qualifying.csv")

# 2. TRANSFORM
qualifying_silver_df = qualifying_df \
    .withColumnRenamed("qualifyId", "qualify_id") \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumn("ingestion_date", current_timestamp())

# 3. WRITE
qualifying_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_qualifying")

display(qualifying_silver_df)

In [0]:
# 1. READ the existing CSV
df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/constructors.csv")

# 2. WRITE it out as JSON
# Note: Spark writes "directories" of data, not just single files.
# We will save this to a new folder called 'json_data'
df_csv.write.mode("overwrite").json("/Volumes/workspace/default/raw_data/json_data/constructors")

print("Conversion Successful! ✅")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/json_data/constructors"))

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. READ from the new JSON folder
# Spark is smart enough to read the whole folder as one dataset
constructors_json_df = spark.read.format("json") \
    .load("/Volumes/workspace/default/raw_data/json_data/constructors")

# 2. TRANSFORM (Same logic as before)
constructors_silver_df = constructors_json_df \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("constructorRef", "constructor_ref") \
    .withColumn("ingestion_date", current_timestamp()) \
    .drop("url")

# 3. WRITE to Silver
constructors_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_constructors")

display(constructors_silver_df)